In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# MODEL = [  # owner = which teammate runs it
#     dict(repo="openai-community/gpt2",                 cat="1 Very Tiny (<0.5B)",  params_b=0.124, owner="1"),
#     dict(repo="google/flan-t5-large",                  cat="2 Tiny (0.5-1B)",      params_b=0.78,  owner="3"),
#     dict(repo="Qwen/Qwen2.5-1.5B-Instruct",            cat="3 Small (1-3B)",       params_b=1.5,   owner="3"),
#     dict(repo="nvidia/Minitron-4B-Base",               cat="4 Medium (3-5B)",      params_b=4.0,   owner="3"),
#     dict(repo="01-ai/Yi-1.5-6B-Chat",                  cat="5 Medium-Big (5-7B)",  params_b=6.1,   owner="1"),
model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.config.pad_token_id = tokenizer.eos_token_id

model.eval()

print("Model ready.")

def generate_text(prompt, max_new_tokens=60, temperature=0.9, do_sample=True):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2169.61it/s, Materializing param=transformer.wte.weight]             

Model ready.


In [3]:
load_dotenv()

# Retrieve the Hugging Face access token.
HF_TOKEN = os.getenv("HF_TOKEN")

# Verify that the token was found without displaying the secret.
if HF_TOKEN:
    print("✓ Hugging Face token loaded successfully.")
else:
    print("✗ Hugging Face token not found.")

✓ Hugging Face token loaded successfully.


In [ ]:
# ---------------------------------------------------------
# ZERO-SHOT PROMPTING
# No examples provided
# ---------------------------------------------------------

prompt = "Sentiment (Positive or Negative): I love this framework →"

response = generate_text(
    prompt,
    max_new_tokens=5,   # keep small so it doesn't continue pattern
    do_sample=False     # deterministic for stability
)

print(response)

In [ ]:
# ---------------------------------------------------------
# ONE-SHOT PROMPTING
# One example provided
# ---------------------------------------------------------

prompt = """
I hate this product → Negative
This is amazing → Positive
I love this framework →
"""

response = generate_text(
    prompt,
    max_new_tokens=5,   # keep small to avoid pattern continuation
    do_sample=False     # deterministic
)

# Print only the generated continuation
print(response[len(prompt):])

In [ ]:
# ---------------------------------------------------------
# FEW-SHOT PROMPTING
# Multiple examples reinforce pattern
# ---------------------------------------------------------

prompt = """
I hate this product → Negative
This is amazing → Positive
The experience was terrible → Negative
I love this framework →
"""

response = generate_text(
    prompt,
    max_new_tokens=5,   # keep small so it doesn't keep generating more lines
    do_sample=False     # deterministic output
)

# Print only the continuation (not the full prompt)
print(response[len(prompt):])